<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

[ваш текст]

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) создайте явную реализации интерфейса и управление зависимостями 


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [5]:
public interface IBankAccount
{
    string AccountNumber { get; }
    string Currency { get; }

    void GetInfo();
    void Deposit(decimal amount);
    void Withdraw(decimal amount);
    void CalculateFees();
    void FreezeAccount();
    void ActivateAccount();
    decimal ConvertToCurrency(decimal amount, string targetCurrency);
}


public class BankAccount : IBankAccount
{
    public string AccountNumber { get; set; }
    public string AccountType { get; set; }
    public decimal Balance { get; set; }

    public DateTime CreatedDate { get; set; }
    public string Currency { get; set; }
    public string OwnerName { get; set; }
    public bool IsActive { get; set; }
    public decimal MinimumBalance { get; set; }

    public BankAccount(string accNum, string accTp, decimal val, string currency = "USD", string owner = "Unknown")
    {
        AccountNumber = accNum;
        AccountType = accTp;
        Balance = val;
        CreatedDate = DateTime.Now;
        Currency = currency;
        OwnerName = owner;
        IsActive = true;
        MinimumBalance = 0;
    }

    public virtual void GetInfo()
    {
        Console.WriteLine($"Account number: {AccountNumber}");
        Console.WriteLine($"Account type: {AccountType}");
        Console.WriteLine($"Balance: {Balance} {Currency}");
        Console.WriteLine($"Owner: {OwnerName}");
        Console.WriteLine($"Created: {CreatedDate}");
        Console.WriteLine($"Status: {(IsActive ? "Active" : "Inactive")}");
        Console.WriteLine($"Minimum balance: {MinimumBalance} {Currency}");
    }

    public virtual void Deposit(decimal amount)
    {
        if (!IsActive)
        {
            Console.WriteLine("Cannot deposit to inactive account.");
            return;
        }
        
        Balance += amount;
        Console.WriteLine($"{amount} {Currency} has been deposited to account {AccountNumber}. New balance: {Balance} {Currency}.");
    }

    public virtual void Withdraw(decimal amount)
    {
        if (!IsActive)
        {
            Console.WriteLine("Cannot withdraw from inactive account.");
            return;
        }
        
        if (Balance - amount >= MinimumBalance)
        {
            Balance -= amount;
            Console.WriteLine($"{amount} {Currency} has been withdrawn from account {AccountNumber}. New balance: {Balance} {Currency}.");
        }
        else
        {
            Console.WriteLine($"Insufficient funds. Minimum balance requirement: {MinimumBalance} {Currency}.");
        }
    }

    public virtual void CalculateFees()
    {
        decimal monthlyFee = 5.0m;
        if (Balance > 1000)
        {
            Console.WriteLine($"No monthly fee applied. Balance is above threshold.");
        }
        else
        {
            Balance -= monthlyFee;
            Console.WriteLine($"Monthly fee of {monthlyFee} {Currency} applied. New balance: {Balance} {Currency}.");
        }
    }

    public virtual void FreezeAccount()
    {
        IsActive = false;
        Console.WriteLine($"Account {AccountNumber} has been frozen.");
    }

    public virtual void ActivateAccount()
    {
        IsActive = true;
        Console.WriteLine($"Account {AccountNumber} has been activated.");
    }

    public virtual decimal ConvertToCurrency(decimal amount, string targetCurrency)
    {
        var exchangeRates = new Dictionary<string, decimal>
        {
            { "USD", 1.0m },
            { "EUR", 0.85m },
            { "GBP", 0.73m },
            { "JPY", 110.0m }
        };

        if (exchangeRates.ContainsKey(Currency) && exchangeRates.ContainsKey(targetCurrency))
        {
            decimal rate = exchangeRates[targetCurrency] / exchangeRates[Currency];
            return amount * rate;
        }
        
        Console.WriteLine($"Currency conversion not supported: {Currency} -> {targetCurrency}");
        return amount;
    }
}


public class SavingsAccount : BankAccount, IBankAccount
{
    public decimal InterestRate { get; set; }
    public decimal MonthlyDepositLimit { get; set; }
    private decimal monthlyDeposits;

    public SavingsAccount(string accNum, decimal val, decimal rate, decimal depositLimit = 10000) 
        : base(accNum, "Savings", val, "USD", "Savings Customer")
    {
        InterestRate = rate;
        MonthlyDepositLimit = depositLimit;
        monthlyDeposits = 0;
        MinimumBalance = 50;
    }

    public override void Deposit(decimal amount)
    {
        if (!IsActive)
        {
            Console.WriteLine("Cannot deposit to inactive account.");
            return;
        }

        if (monthlyDeposits + amount > MonthlyDepositLimit)
        {
            Console.WriteLine($"Deposit limit exceeded. Monthly limit: {MonthlyDepositLimit} {Currency}");
            return;
        }

        decimal interest = amount * InterestRate;
        base.Deposit(interest + amount);
        monthlyDeposits += amount;
        Console.WriteLine($"Interest accrued: {interest} {Currency}.");
        Console.WriteLine($"Monthly deposits total: {monthlyDeposits} {Currency} of {MonthlyDepositLimit} {Currency} limit.");
    }

    public override void GetInfo()
    {
        base.GetInfo();
        Console.WriteLine($"Interest rate: {InterestRate:P2}");
        Console.WriteLine($"Monthly deposit limit: {MonthlyDepositLimit} {Currency}");
        Console.WriteLine($"Current monthly deposits: {monthlyDeposits} {Currency}");
    }

    void IBankAccount.CalculateFees()
    {
        if (Balance < 100)
        {
            decimal lowBalanceFee = 2.0m;
            Balance -= lowBalanceFee;
            Console.WriteLine($"Low balance fee of {lowBalanceFee} {Currency} applied to savings account.");
        }
        else
        {
            Console.WriteLine("No fees applied to savings account.");
        }
    }

    void IBankAccount.FreezeAccount()
    {
        IsActive = false;
        Console.WriteLine($"Savings account {AccountNumber} frozen. Interest accumulation paused.");
    }

    void IBankAccount.ActivateAccount()
    {
        IsActive = true;
        Console.WriteLine($"Savings account {AccountNumber} activated. Interest accumulation resumed.");
    }

    decimal IBankAccount.ConvertToCurrency(decimal amount, string targetCurrency)
    {
        Console.WriteLine($"Currency conversion for savings account with preferential rates.");
        return base.ConvertToCurrency(amount, targetCurrency) * 0.95m;
    }
}


public class CheckingAccount : BankAccount, IBankAccount
{
    public decimal OverdraftLimit { get; set; }
    public int MonthlyFreeTransactions { get; set; }
    private int transactionCount;

    public CheckingAccount(string accNum, decimal val, decimal lim, int freeTransactions = 10) 
        : base(accNum, "Checking", val, "USD", "Checking Customer")
    {
        OverdraftLimit = lim;
        MonthlyFreeTransactions = freeTransactions;
        transactionCount = 0;
    }

    public override void Withdraw(decimal amount)
    {
        if (!IsActive)
        {
            Console.WriteLine("Cannot withdraw from inactive account.");
            return;
        }

        transactionCount++;
        ApplyTransactionFee();

        if (Balance + OverdraftLimit >= amount)
        {
            base.Withdraw(amount);
            if (Balance < 0) 
                Console.WriteLine($"Warning! Account is overdrawn. Available credit: {Balance + OverdraftLimit} {Currency}.");
        }
        else
        {
            Console.WriteLine("Transaction declined! Overdraft limit exceeded.");
        }
    }

    private void ApplyTransactionFee()
    {
        if (transactionCount > MonthlyFreeTransactions)
        {
            decimal transactionFee = 1.0m;
            Balance -= transactionFee;
            Console.WriteLine($"Transaction fee of {transactionFee} {Currency} applied. Transaction #{transactionCount}");
        }
    }

    public override void GetInfo()
    {
        base.GetInfo();
        Console.WriteLine($"Overdraft limit: {OverdraftLimit} {Currency}");
        Console.WriteLine($"Monthly free transactions: {MonthlyFreeTransactions}");
        Console.WriteLine($"Transactions this month: {transactionCount}");
        Console.WriteLine($"Remaining free transactions: {Math.Max(0, MonthlyFreeTransactions - transactionCount)}");
    }

    void IBankAccount.CalculateFees()
    {
        decimal monthlyFee = 10.0m;
        if (Balance > 5000)
        {
            monthlyFee = 0;
        }

        if (monthlyFee > 0)
        {
            Balance -= monthlyFee;
            Console.WriteLine($"Monthly checking account fee of {monthlyFee} {Currency} applied.");
        }
        else
        {
            Console.WriteLine("No monthly fee for checking account (balance requirement met).");
        }
    }

    void IBankAccount.FreezeAccount()
    {
        IsActive = false;
        Console.WriteLine($"Checking account {AccountNumber} frozen. All transactions blocked.");
    }

    void IBankAccount.ActivateAccount()
    {
        IsActive = true;
        Console.WriteLine($"Checking account {AccountNumber} activated. Transactions enabled.");
    }

    decimal IBankAccount.ConvertToCurrency(decimal amount, string targetCurrency)
    {
        Console.WriteLine($"Currency conversion for checking account with standard rates.");
        return base.ConvertToCurrency(amount, targetCurrency);
    }
}


public class InvestmentAccount : BankAccount, IBankAccount
{
    public List<string> AssetsList { get; set; }
    public decimal RiskLevel { get; set; }
    public decimal ManagementFee { get; set; }

    public InvestmentAccount(string accNum, decimal val, List<string> asL, decimal risk = 0.5m, decimal mgmtFee = 0.01m) 
        : base(accNum, "Investment", val, "USD", "Investment Client")
    {
        AssetsList = asL;
        RiskLevel = risk;
        ManagementFee = mgmtFee;
        MinimumBalance = 1000;
    }

    public override void GetInfo()
    {
        string list = string.Join(", ", AssetsList);
        base.GetInfo();
        Console.WriteLine($"Assets list: {list}");
        Console.WriteLine($"Risk level: {RiskLevel:P2}");
        Console.WriteLine($"Management fee: {ManagementFee:P2}");
    }

    public void AddAsset(string asset)
    {
        AssetsList.Add(asset);
        Console.WriteLine($"Added asset: {asset} to account {AccountNumber}.");
    }

    public void RemoveAsset(string asset)
    {
        if (AssetsList.Contains(asset))
        {
            AssetsList.Remove(asset);
            Console.WriteLine($"Removed asset: {asset} from account {AccountNumber}.");
        }
        else
        {
            Console.WriteLine($"Asset {asset} not found on account {AccountNumber}.");
        }
    }

    public void CalculatePerformance()
    {
        decimal performance = Balance * RiskLevel * 0.1m;
        Balance += performance;
        Console.WriteLine($"Investment performance: {performance} {Currency}. New balance: {Balance} {Currency}");
    }

    void IBankAccount.CalculateFees()
    {
        decimal fee = Balance * ManagementFee;
        Balance -= fee;
        Console.WriteLine($"Investment management fee of {fee} {Currency} ({ManagementFee}) applied.");
    }

    void IBankAccount.FreezeAccount()
    {
        IsActive = false;
        Console.WriteLine($"Investment account {AccountNumber} frozen. Trading suspended.");
    }

    void IBankAccount.ActivateAccount()
    {
        IsActive = true;
        Console.WriteLine($"Investment account {AccountNumber} activated. Trading enabled.");
    }

    decimal IBankAccount.ConvertToCurrency(decimal amount, string targetCurrency)
    {
        Console.WriteLine($"Currency conversion for investment account with institutional rates.");
        return base.ConvertToCurrency(amount, targetCurrency) * 0.98m;
    }
}

public class AccountManager
{
    private readonly IBankAccount _bankAccount;

    public AccountManager(IBankAccount bankAccount)
    {
        _bankAccount = bankAccount;
    }

    public void PerformAccountOperations()
    {
        Console.WriteLine($"\n=== Performing operations on account {_bankAccount.AccountNumber} ===");
        
        _bankAccount.GetInfo();
        Console.WriteLine();
        
        _bankAccount.Deposit(500);
        Console.WriteLine();
        
        _bankAccount.Withdraw(200);
        Console.WriteLine();
        
        _bankAccount.CalculateFees();
        Console.WriteLine();

        decimal convertedAmount = _bankAccount.ConvertToCurrency(100, "EUR");
        Console.WriteLine($"Converted 100 {_bankAccount.Currency} to EUR: {convertedAmount}");
        Console.WriteLine();
        
        _bankAccount.GetInfo();
    }

    public void ManageAccountStatus(bool activate)
    {
        if (activate)
            _bankAccount.ActivateAccount();
        else
            _bankAccount.FreezeAccount();
    }
}

SavingsAccount savings = new SavingsAccount("SAV001", 1000, 0.02m, 5000);
InvestmentAccount investment = new InvestmentAccount("INV001", 5000, 
            new List<string> { "APPLE", "GOOGLE", "MICROSOFT" }, 0.7m, 0.015m);

AccountManager savManager = new AccountManager(savings);
savManager.PerformAccountOperations();

Console.WriteLine();

AccountManager invManager = new AccountManager(investment);
invManager.PerformAccountOperations();


=== Performing operations on account SAV001 ===
Account number: SAV001
Account type: Savings
Balance: 1000 USD
Owner: Savings Customer
Created: 11/2/2025 1:22:08 PM
Status: Active
Minimum balance: 50 USD
Interest rate: 2.00%
Monthly deposit limit: 5000 USD
Current monthly deposits: 0 USD

510.00 USD has been deposited to account SAV001. New balance: 1510.00 USD.
Interest accrued: 10.00 USD.
Monthly deposits total: 500 USD of 5000 USD limit.

200 USD has been withdrawn from account SAV001. New balance: 1310.00 USD.

No fees applied to savings account.

Currency conversion for savings account with preferential rates.
Converted 100 USD to EUR: 80.7500

Account number: SAV001
Account type: Savings
Balance: 1310.00 USD
Owner: Savings Customer
Created: 11/2/2025 1:22:08 PM
Status: Active
Minimum balance: 50 USD
Interest rate: 2.00%
Monthly deposit limit: 5000 USD
Current monthly deposits: 500 USD


=== Performing operations on account INV001 ===
Account number: INV001
Account type: Investme